# Gikomba Market Stall Performance Analysis

**Analyst:** Anne Mugunye · PLP Data Analytics 1 Million Devs Cohort

**Scenario:** Wanjiku runs a fresh-produce stall at Gikomba Market. She has two weeks of daily sales and a rough stock record, and she wants to know: *How is my business doing? Is it growing? What should I restock, and where should I focus?*

**Approach:** I follow the analytics lifecycle (Ask, Prepare, Process, Analyse, Share, Act) using core Python only: lists, dictionaries, loops and functions. No pandas or NumPy.

In [1]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

## Task 1: Summary statistics for the 14 days

**Approach:** `sum()`, `max()` and `min()` give the total and the extremes, and the average is the total divided by the number of days. `list.index()` returns the position of the best and worst day. Positions 0 to 6 are week 1, so `position // 7 + 1` gives the week number, and the same position in the `days` list gives the day name.

In [2]:
total_sales = sum(daily_sales)
average_sales = total_sales / len(daily_sales)
highest = max(daily_sales)
lowest = min(daily_sales)

best_pos = daily_sales.index(highest)
worst_pos = daily_sales.index(lowest)
best_day, best_week = days[best_pos], best_pos // 7 + 1
worst_day, worst_week = days[worst_pos], worst_pos // 7 + 1

print(f"Total sales:   KES {total_sales:,}")
print(f"Average/day:   KES {average_sales:,.2f}")
print(f"Highest day:   KES {highest:,} ({best_day}, week {best_week})")
print(f"Lowest day:    KES {lowest:,} ({worst_day}, week {worst_week})")

Total sales:   KES 46,450
Average/day:   KES 3,317.86
Highest day:   KES 5,600 (Sat, week 2)
Lowest day:    KES 1,950 (Tue, week 1)


## Task 2: Weekly totals and week-on-week growth

**Approach:** The function `week_totals()` slices the list in two (`[:7]` is week 1, `[7:]` is week 2) and returns both totals as a tuple. I unpack the tuple into two variables and calculate growth as (week 2 − week 1) ÷ week 1 × 100.

In [3]:
def week_totals(sales):
    """Return (week 1 total, week 2 total) for a list of 14 daily sales."""
    return sum(sales[:7]), sum(sales[7:])


week1_total, week2_total = week_totals(daily_sales)
growth_pct = (week2_total - week1_total) / week1_total * 100

print(f"Week 1 total: KES {week1_total:,}")
print(f"Week 2 total: KES {week2_total:,}")
print(f"Week-on-week growth: {growth_pct:.1f}%")

Week 1 total: KES 22,050
Week 2 total: KES 24,400
Week-on-week growth: 10.7%


## Task 3: Classifying each day as Strong, Normal or Weak

**Approach:** `classify_day()` compares one day's sales with the average. More than 20% above the average (average × 1.2) is "Strong", more than 20% below (average × 0.8) is "Weak", and anything in between is "Normal". I apply it to every day and count the results with the dictionary counting pattern `counts.get(label, 0) + 1`.

In [4]:
def classify_day(amount, average):
    """Return "Strong" (>20% above average), "Weak" (>20% below) or "Normal"."""
    if amount > average * 1.2:
        return "Strong"
    elif amount < average * 0.8:
        return "Weak"
    else:
        return "Normal"


day_counts = {}
for day, amount in zip(days, daily_sales):
    label = classify_day(amount, average_sales)
    day_counts[label] = day_counts.get(label, 0) + 1
    print(f"{day}: KES {amount:>5,} -> {label}")

print(day_counts)

Mon: KES 2,300 -> Weak
Tue: KES 1,950 -> Weak
Wed: KES 2,800 -> Normal
Thu: KES 2,100 -> Weak
Fri: KES 3,600 -> Normal
Sat: KES 5,200 -> Strong
Sun: KES 4,100 -> Strong
Mon: KES 2,500 -> Weak
Tue: KES 2,200 -> Weak
Wed: KES 3,100 -> Normal
Thu: KES 2,400 -> Weak
Fri: KES 3,900 -> Normal
Sat: KES 5,600 -> Strong
Sun: KES 4,700 -> Strong
{'Weak': 6, 'Normal': 4, 'Strong': 4}


## Task 4: Weekdays versus weekends

**Approach:** I loop through the days and the sales together. Saturday and Sunday go into a weekend list and the other days go into a weekday list. Then I compare the two averages: the weekend average minus the weekday average, divided by the weekday average.

In [5]:
weekday_sales = []
weekend_sales = []
for day, amount in zip(days, daily_sales):
    if day in ("Sat", "Sun"):
        weekend_sales.append(amount)
    else:
        weekday_sales.append(amount)

weekday_avg = sum(weekday_sales) / len(weekday_sales)
weekend_avg = sum(weekend_sales) / len(weekend_sales)
weekend_lift_pct = (weekend_avg - weekday_avg) / weekday_avg * 100

print(f"Weekday average (Mon-Fri): KES {weekday_avg:,.2f}")
print(f"Weekend average (Sat-Sun): KES {weekend_avg:,.2f}")
print(f"Weekends sell {weekend_lift_pct:.1f}% more than weekdays")

Weekday average (Mon-Fri): KES 2,685.00
Weekend average (Sat-Sun): KES 4,900.00
Weekends sell 82.5% more than weekdays


## Task 5: Stock to reorder and gross margins

**Approach:** `products_to_reorder()` loops over `stock.items()` and collects the names where the quantity is at or below the reorder level. A second small function, `gross_margin_pct()`, calculates (price − cost) ÷ price × 100 for one product, so the formula is written only once and reused for every product.

In [6]:
def products_to_reorder(items):
    """Return a list of product names whose stock is at or below the reorder level."""
    low_stock = []
    for name, info in items.items():
        if info["qty"] <= info["reorder"]:
            low_stock.append(name)
    return low_stock


def gross_margin_pct(info):
    """Return the gross margin percentage: (price - cost) / price * 100."""
    return (info["price"] - info["cost"]) / info["price"] * 100


to_reorder = products_to_reorder(stock)
margins = {}
for name, info in stock.items():
    margins[name] = gross_margin_pct(info)

print("Reorder now:", to_reorder)
for name, margin in margins.items():
    print(f"{name}: {margin:.1f}%")

Reorder now: ['Tomatoes', 'Sukuma']
Tomatoes: 33.3%
Onions: 30.0%
Sukuma: 40.0%
Potatoes: 33.3%
Avocado: 42.9%


## Task 6: Stall Performance Report

**Approach:** f-strings print the report. `:,` adds thousands separators, `.2f` shows two decimals, and `:<10` and `:>5` align the columns.

In [7]:
line = "=" * 46
print(line)
print("      STALL PERFORMANCE REPORT - GIKOMBA")
print(line)
print(f"Period:               14 days")
print(f"Total sales:          KES {total_sales:,}")
print(f"Average per day:      KES {average_sales:,.2f}")
print(f"Best day:             {best_day}, week {best_week} (KES {highest:,})")
print(f"Worst day:            {worst_day}, week {worst_week} (KES {lowest:,})")
print()
print(f"Week 1 total:         KES {week1_total:,}")
print(f"Week 2 total:         KES {week2_total:,}")
print(f"Week-on-week growth:  {growth_pct:.1f}%")
print()
print(f"Weekday average:      KES {weekday_avg:,.2f}")
print(f"Weekend average:      KES {weekend_avg:,.2f}")
print(f"Weekend difference:   +{weekend_lift_pct:.1f}%")
print()
print("Day classification:")
for label in ("Strong", "Normal", "Weak"):
    print(f"  {label:<8}{day_counts.get(label, 0):>2} days")
print()
print(f"{'Product':<10}{'Stock':>7}{'Reorder':>9}{'Margin':>9}  Status")
for name, info in stock.items():
    status = "REORDER" if name in to_reorder else "OK"
    print(f"{name:<10}{info['qty']:>5}kg{info['reorder']:>7}kg{margins[name]:>8.1f}%  {status}")
print(line)

      STALL PERFORMANCE REPORT - GIKOMBA
Period:               14 days
Total sales:          KES 46,450
Average per day:      KES 3,317.86
Best day:             Sat, week 2 (KES 5,600)
Worst day:            Tue, week 1 (KES 1,950)

Week 1 total:         KES 22,050
Week 2 total:         KES 24,400
Week-on-week growth:  10.7%

Weekday average:      KES 2,685.00
Weekend average:      KES 4,900.00
Weekend difference:   +82.5%

Day classification:
  Strong   4 days
  Normal   4 days
  Weak     6 days

Product     Stock  Reorder   Margin  Status
Tomatoes     18kg     25kg    33.3%  REORDER
Onions       40kg     20kg    30.0%  OK
Sukuma        9kg     15kg    40.0%  REORDER
Potatoes     55kg     30kg    33.3%  OK
Avocado      12kg     10kg    42.9%  OK


## Task 7: Recommendations for Wanjiku

1. **Your business is growing.** Sales rose from KES 22,050 in week 1 to KES 24,400 in week 2, which is 10.7% growth, so keep your current suppliers and pricing.
2. **Focus on weekends.** Saturday and Sunday average KES 4,900 against KES 2,685 on weekdays (82.5% more), and all 4 "Strong" days were weekends, so stock up before Friday and have enough help on Saturday and Sunday.
3. **Lift your weak weekdays.** 6 of the 14 days were "Weak" (every Monday, Tuesday and Thursday), and Tuesday of week 1 was the lowest at KES 1,950, so try a weekday offer such as a tomato and onion bundle.
4. **Restock Tomatoes and Sukuma now.** Tomatoes are at 18 kg against a reorder level of 25 kg, and Sukuma is at 9 kg against 15 kg, so order both before the weekend rush.
5. **Push Avocado and avoid over-buying Potatoes.** Avocado has the best margin at 42.9% and only 12 kg in stock (reorder level 10 kg), while Potatoes have 55 kg against a reorder level of 30 kg at a 33.3% margin, so spend your next order on avocados.